# Intégration MLflow — Prédiction de Gravité des Accidents Routiers

Ce notebook intègre MLflow dans le pipeline de ML pour la prédiction de gravité des accidents (données BAAC).

##  Plan du notebook
1. Imports & Configuration MLflow
2. Chargement & Préparation des données
3. Baseline — Comparaison des 4 modèles
4. Tuning manuel — 3 configurations XGBoost
5. Artefacts visuels — Confusion matrix & Courbes ROC
6. GridSearchCV — Recherche exhaustive
7. Optuna — Optimisation bayésienne
8. Model Registry — Enregistrement du meilleur modèle
9. Comparaison finale des approches

---
## 1.  Imports & Configuration MLflow

> **Rappel architecture MLflow** :
> - `mlflow.set_tracking_uri` → adresse du serveur MLflow
> - `mlflow.set_experiment` → dossier logique regroupant les runs
> - Règle : **1 objectif = 1 expérience**

In [2]:
# === Librairies standard ===
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')  # Evite les problèmes d'affichage hors Jupyter
import matplotlib.pyplot as plt

# === Machine Learning ===
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    precision_score, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# === MLflow ===
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
from mlflow import MlflowClient

# === Optuna ===
import optuna
from optuna.integration.mlflow import MLflowCallback

# === Configuration MLflow ===
# Tracking URI : adresse où MLflow envoie les données
TRACKING_URI = "http://127.0.0.1:5000"
mlflow.set_tracking_uri(TRACKING_URI)

print(f"✅ MLflow configuré → {mlflow.get_tracking_uri()}")

c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ MLflow configuré → http://127.0.0.1:5000


---
## 2. Chargement & Préparation des données

In [3]:
# Chargement du dataset BAAC
df = pd.read_csv("../data/raw/accidents.csv")
print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head(3)

Dataset chargé : 506467 lignes, 34 colonnes


,grav,sexe,trajet,secu1,senc,obsm,choc,manv,motor,lum,...,nuit_sans_eclairage,surface_dangereuse,vitesse_x_collision,age_x_securite,agglo_x_vitesse,saison,est_mois_vacances,deux_roues,poids_lourd,vehicule_leger
0,3,1.0,1.0,1.0,1.0,2.0,1.0,1.0,5.0,2.0,...,0,0,80.0,0.0,80.0,3.0,0,0,0,0
1,1,1.0,1.0,1.0,1.0,9.0,3.0,17.0,1.0,2.0,...,0,0,80.0,43.0,80.0,3.0,0,0,0,1
2,4,1.0,5.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,...,0,0,0.0,38.0,80.0,3.0,0,0,0,1


In [4]:
# Séparation features / cible
X = df.drop(columns=['grav'])
y = df['grav']

# ⚠️ Correction classes XGBoost : [1,2,3,4] → [0,1,2,3]
# XGBoost exige que les classes commencent à 0
y = y - 1

# Train/Test split stratifié (conserve les proportions de classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train : {X_train.shape[0]} exemples")
print(f"Test  : {X_test.shape[0]} exemples")
print(f"Classes : {sorted(y.unique())}")

Train : 405173 exemples
Test  : 101294 exemples
Classes : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


---
## 3.  Baseline — Comparaison des 4 modèles

On entraîne 4 modèles avec leurs paramètres par défaut pour établir une **référence de performance**.

> **Choix de la métrique** : `recall` (weighted) car dans un contexte d'accidents routiers,
> minimiser les **faux négatifs sur les cas graves** est prioritaire.
> Un accident grave prédit bénin = des secours qui n'arrivent pas à temps.

> **Autolog** : capture automatiquement hyperparamètres, modèle et feature importance.
> Les métriques sklearn (recall, roc_auc...) sont ajoutées manuellement car autolog
> ne les capture pas (elles sont calculées après le fit).

In [6]:
mlflow.set_experiment("accidents-gravite-test")

# On ouvre un "run" = une session d'enregistrement
with mlflow.start_run(run_name="XGBoost-baseline"):
    
    # 1. Entraînement (ton code existant)
    model = XGBClassifier(max_depth=3, n_estimators=100, learning_rate=0.1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    # 2. Logger les hyperparamètres
    mlflow.log_params(model.get_params())
    
    # 3. Logger les métriques
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("f1", f1_score(y_test, y_pred, average="weighted"))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba,  average='weighted', multi_class='ovr'))
    mlflow.log_metric("precision", precision_score(y_test, y_pred, average="weighted"))
    mlflow.log_metric("recall", recall_score(y_test, y_pred, average="weighted"))
    
    # 4. Logger le modèle sérialisé
    mlflow.xgboost.log_model(model, artifact_path="model")

2026/02/25 21:49:03 INFO mlflow.tracking.fluent: Experiment with name 'accidents-gravite-test' does not exist. Creating a new experiment.
2026/02/25 21:49:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
202

🏃 View run XGBoost-baseline at: http://127.0.0.1:5000/#/experiments/8/runs/1dd9b5d4b8ae4944afb1238067b9dbc6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/8


In [7]:
# Activation de l'autolog pour chaque librairie
mlflow.sklearn.autolog()
mlflow.xgboost.autolog()
mlflow.lightgbm.autolog()

# Expérience dédiée à la comparaison des algorithmes
mlflow.set_experiment("accidents-gravite-baseline")

# Dictionnaire modèles : nom → (instance, module mlflow)
models = {
    "LogisticRegression": (LogisticRegression(max_iter=100), mlflow.sklearn),
    "RandomForest":       (RandomForestClassifier(n_estimators=20, random_state=42), mlflow.sklearn),
    "XGBoost":            (XGBClassifier(n_estimators=100, random_state=42), mlflow.xgboost),
    "LightGBM":           (LGBMClassifier(n_estimators=100, random_state=42), mlflow.lightgbm),
}

for run_name, (model, mlflow_module) in models.items():
    with mlflow.start_run(run_name=run_name):

        # Entraînement
        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)  # toutes les colonnes (multiclasse)

        # Métriques sklearn — average='weighted' car classes déséquilibrées
        mlflow.log_metric("recall",    recall_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("roc_auc",   roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))
        mlflow.log_metric("f1",        f1_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("precision", precision_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("accuracy",  accuracy_score(y_test, y_pred))

        print(f"✅ {run_name} loggé — recall={recall_score(y_test, y_pred, average='weighted'):.4f}")

2026/02/25 21:50:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to conver

✅ LogisticRegression loggé — recall=0.5742
🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/7/runs/a18a7a45706147f4b4fc52f8346df660
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


2026/02/25 21:51:49 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/02/25 21:52:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\

✅ RandomForest loggé — recall=0.6410
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/7/runs/a825046f817241e7b17ff603137ff7e5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


2026/02/25 21:53:37 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/02/25 21:53:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 21:53:52 WARNING mlflow

✅ XGBoost loggé — recall=0.6608
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/7/runs/967f68751e3946c49575e68099021266
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


2026/02/25 21:54:23 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037961 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 469
[LightGBM] [Info] Number of data points in the train set: 405173, number of used features: 33
[LightGBM] [Info] Start training from score -0.856395
[LightGBM] [Info] Start training from score -3.617480
[LightGBM] [Info] Start training from score -1.886904
[LightGBM] [Info] Start training from score -0.924022


2026/02/25 21:54:36 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/02/25 21:54:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 21:54:36 WARNING mlflow

✅ LightGBM loggé — recall=0.6578
🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/7/runs/823c2fe0efec4992ac6f185fee634d68
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


---
## 4. Tuning manuel — 3 configurations XGBoost

On teste manuellement 3 configurations pour comprendre l'impact des hyperparamètres.
Chaque configuration = 1 run MLflow distinct dans une **expérience séparée**.

| Config | Philosophie |
|---|---|
| Config 1 | Modèle simple et rapide |
| Config 2 | Équilibre complexité/vitesse |
| Config 3 | Modèle complexe (risque overfitting) |

In [8]:
# Expérience dédiée au tuning manuel
mlflow.set_experiment("accidents-gravite-tuning")

configs = [
    {"max_depth": 3, "n_estimators": 100, "learning_rate": 0.1,  "subsample": 1.0},
    {"max_depth": 5, "n_estimators": 200, "learning_rate": 0.05, "subsample": 0.8},
    {"max_depth": 7, "n_estimators": 300, "learning_rate": 0.01, "subsample": 0.8},
]

for i, params in enumerate(configs):
    with mlflow.start_run(run_name=f"XGBoost-config-{i+1}"):

        model = XGBClassifier(**params, random_state=42)
        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        mlflow.log_params(params)
        mlflow.log_metric("recall",    recall_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("roc_auc",   roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))
        mlflow.log_metric("f1",        f1_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("precision", precision_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("accuracy",  accuracy_score(y_test, y_pred))

        print(f"✅ Config {i+1} loggée — recall={recall_score(y_test, y_pred, average='weighted'):.4f}")

2026/02/25 21:56:03 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/02/25 21:56:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 21:56:18 WARNING mlflow

✅ Config 1 loggée — recall=0.6329
🏃 View run XGBoost-config-1 at: http://127.0.0.1:5000/#/experiments/2/runs/9f41369c2cd94a859a13663868b2c3ca
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/02/25 21:57:03 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/02/25 21:57:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 21:57:16 WARNING mlflow

✅ Config 2 loggée — recall=0.6500
🏃 View run XGBoost-config-2 at: http://127.0.0.1:5000/#/experiments/2/runs/d4e5cc6cd95c4f6bb60b2b63cf4c794d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/02/25 21:58:46 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/02/25 21:58:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 21:59:03 WARNING mlflow

✅ Config 3 loggée — recall=0.6444
🏃 View run XGBoost-config-3 at: http://127.0.0.1:5000/#/experiments/2/runs/1e09a1ba0c4e468294a6e1da05806de6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


---
## 5. Artefacts visuels — Confusion Matrix & Courbes ROC

On logue des artefacts visuels pour enrichir l'analyse dans l'UI MLflow.

> **Workflow obligatoire** : sauvegarder sur disque → `log_artifact()`
> MLflow ne peut pas logger un graphique matplotlib directement en mémoire.

In [10]:
# Dossier temporaire pour les artefacts
os.makedirs("tmp_artifacts", exist_ok=True)

mlflow.set_experiment("accidents-gravite-tuning")

# Meilleure config identifiée : Config 2
best_params = {"max_depth": 5, "n_estimators": 200, "learning_rate": 0.05, "subsample": 0.8}

with mlflow.start_run(run_name="XGBoost-config2-artefacts"):

    model = XGBClassifier(**best_params, random_state=42)
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    # --- Paramètres & métriques ---
    mlflow.log_params(best_params)
    mlflow.log_metric("recall",    recall_score(y_test, y_pred, average='weighted'))
    mlflow.log_metric("roc_auc",   roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))
    mlflow.log_metric("f1",        f1_score(y_test, y_pred, average='weighted'))

    # --- Artefact 1 : Matrice de confusion ---
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
    ax.set_title("Matrice de confusion — XGBoost Config 2")
    fig.savefig("tmp_artifacts/confusion_matrix.png")
    plt.close(fig)
    mlflow.log_artifact("tmp_artifacts/confusion_matrix.png")

    # --- Artefact 2 : Courbes ROC par classe ---
    classes     = sorted(y_test.unique())
    y_test_bin  = label_binarize(y_test, classes=classes)
    fig, ax     = plt.subplots(figsize=(8, 6))
    for i, classe in enumerate(classes):
        RocCurveDisplay.from_predictions(
            y_test_bin[:, i], y_proba[:, i],
            name=f"Classe {classe}", ax=ax
        )
    ax.set_title("Courbes ROC par classe — XGBoost Config 2")
    fig.savefig("tmp_artifacts/roc_curves.png")
    plt.close(fig)
    mlflow.log_artifact("tmp_artifacts/roc_curves.png")

    # --- Artefact 3 : Feature names ---
    with open("tmp_artifacts/feature_names.txt", "w") as f:
        f.write("\n".join(X_train.columns.tolist()))
    mlflow.log_artifact("tmp_artifacts/feature_names.txt")

    # --- Modèle sérialisé ---
    mlflow.xgboost.log_model(model, artifact_path="model")

    print("✅ Run avec artefacts visuels loggé !")

2026/02/25 22:08:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/02/25 22:08:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 22:08:34 WARNING mlflow

✅ Run avec artefacts visuels loggé !
🏃 View run XGBoost-config2-artefacts at: http://127.0.0.1:5000/#/experiments/2/runs/a0a9453c9c94443c9642ebf2387e4eca
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


####  Meilleure configuration identifiée
Expérience  : accidents-gravite-tuning
 Modèle      : XGBoost
 
 Config retenue :
   - learning_rate : 0.05
   - max_depth     : 5
   - n_estimators  : 200
   - subsample     : 0.8

 Métriques obtenues :
   - Accuracy  : 0.652
   - F1        : 0.631
   - ROC AUC   : 0.827

 Justification :
   Meilleur F1 et ROC AUC parmi les 3 configs testées.
   Config 3 (max_depth=7) plus complexe mais moins performante
   → signe d'overfitting. Config 2 retenue comme meilleur compromis.

 Note : le baseline (params défaut) reste légèrement meilleur en accuracy
        (0.662 vs 0.652). Le tuning GridSearch/Optuna (Jour 3) permettra
        d'explorer plus de combinaisons pour dépasser ce baseline.


---
## 6. GridSearchCV — Recherche exhaustive d'hyperparamètres

GridSearchCV teste **toutes les combinaisons** d'une grille de paramètres.
Avec 3×3×3 = **27 combinaisons**, chaque combinaison est loggée comme un run MLflow distinct.

> **StratifiedKFold** : garantit que chaque fold contient les mêmes proportions de classes
> que le dataset — essentiel avec des classes déséquilibrées.

In [ ]:
mlflow.set_experiment("accidents-gravite-gridsearch")

# Grille de paramètres à explorer
param_grid = {
    "max_depth":     [3, 5, 7],
    "n_estimators":  [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
}

# Cross-validation stratifiée
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=XGBClassifier(random_state=42, subsample=0.8),
    param_grid=param_grid,
    scoring="recall_weighted",  # métrique métier
    cv=cv,
    n_jobs=2,                   # ← 2 cœurs au lieu de tous (-1) car pas assez de RAM
    verbose=1
)
grid_search.fit(X_train, y_train)

# Logger chaque combinaison comme un run MLflow distinct
for i, params in enumerate(grid_search.cv_results_["params"]):
    with mlflow.start_run(run_name=f"GridSearch-{i+1}"):

        model = XGBClassifier(**params, subsample=0.8, random_state=42)
        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        mlflow.log_params(params)
        mlflow.log_metric("recall",    recall_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("roc_auc",   roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))
        mlflow.log_metric("f1",        f1_score(y_test, y_pred, average='weighted'))
        # Score CV — plus fiable que le score sur test fixe
        mlflow.log_metric("cv_recall", grid_search.cv_results_["mean_test_score"][i])
        mlflow.xgboost.log_model(model, artifact_path="model")

print(f"✅ {len(grid_search.cv_results_['params'])} runs GridSearch loggés !")
print(f"   Meilleurs params  : {grid_search.best_params_}")
print(f"   Meilleur CV recall : {grid_search.best_score_:.4f}")

2026/02/25 22:23:48 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'aaff5feed9694be29a983e04a54daf8f', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/02/25 22:24:03 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Mis

Fitting 2 folds for each of 27 candidates, totalling 54 fits


---
## 7. Optuna — Optimisation bayésienne

Optuna apprend des résultats précédents pour orienter intelligemment les prochains essais.
Avantage vs GridSearch : explore des **plages continues** (ex: max_depth entre 3 et 10)
et peut trouver des valeurs que GridSearch n'aurait jamais testées.

> **MLflowCallback** : logue automatiquement chaque trial comme un run MLflow.
> Les métriques sklearn et le modèle sont ajoutés manuellement dans la fonction objective.

In [ ]:
mlflow.set_experiment("accidents-gravite-optuna")
optuna.logging.set_verbosity(optuna.logging.WARNING)  # réduit les logs verbeux

# Callback MLflow — logue automatiquement chaque trial
mlflowcb = MLflowCallback(
    tracking_uri=TRACKING_URI,
    metric_name="recall"
)

@mlflowcb.track_in_mlflow()
def objective(trial):
    """Fonction objective : Optuna cherche les params qui maximisent le recall."""

    # Optuna propose des valeurs dans les plages définies
    params = {
        "max_depth":     trial.suggest_int("max_depth", 3, 10),
        "n_estimators":  trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":     trial.suggest_float("subsample", 0.6, 1.0),
        "random_state":  42
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    # Métriques à logger manuellement (non capturées par MLflowCallback)
    mlflow.log_metric("recall",  recall_score(y_test, y_pred, average='weighted'))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))
    mlflow.log_metric("f1",      f1_score(y_test, y_pred, average='weighted'))

    # Modèle sérialisé — nécessaire pour le Model Registry
    mlflow.xgboost.log_model(model, artifact_path="model")

    return recall_score(y_test, y_pred, average='weighted')


# Lancer l'optimisation — study nommé pour un nom propre dans MLflow
study = optuna.create_study(
    study_name="accidents-gravite-optuna",
    direction="maximize"  # on maximise le recall
)
study.optimize(objective, n_trials=30, callbacks=[mlflowcb])

print(f"\n✅ Optimisation terminée !")
print(f"   Meilleur recall  : {study.best_value:.4f}")
print(f"   Meilleurs params : {study.best_params}")

---
## 8. Model Registry — Enregistrement du meilleur modèle

Le Model Registry est le **catalogue officiel** des modèles validés.
On y enregistre uniquement le meilleur modèle, sélectionné sur le recall.

**Cycle de vie** : `None → Staging → Production → Archived`

In [ ]:
client = MlflowClient()

# Chercher le meilleur run sur TOUTES les expériences
best_run = mlflow.search_runs(
    experiment_names=[
        "accidents-gravite-baseline",
        "accidents-gravite-tuning",
        "accidents-gravite-gridsearch",
        "accidents-gravite-optuna"
    ],
    order_by=["metrics.recall DESC"],
    filter_string="metrics.recall > 0"  # exclut les runs sans recall
).iloc[0]

print(f"🏆 Meilleur run toutes expériences confondues :")
print(f"   Nom    : {best_run['tags.mlflow.runName']}")
print(f"   Recall : {best_run['metrics.recall']:.4f}")
print(f"   Run ID : {best_run['run_id']}")

In [ ]:
# Enregistrer dans le Model Registry
model_uri = f"runs:/{best_run['run_id']}/model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name="xgboost-gravite-accidents"
)

# Ajouter une description
client.update_registered_model(
    name="xgboost-gravite-accidents",
    description="XGBoost pour prédiction gravité accidents routiers. "
                "Sélectionné sur le recall pour minimiser les faux négatifs sur cas graves."
)

client.update_model_version(
    name="xgboost-gravite-accidents",
    version=registered.version,
    description=f"Recall={best_run['metrics.recall']:.4f} — Run: {best_run['tags.mlflow.runName']}"
)

# Passer en Staging
client.transition_model_version_stage(
    name="xgboost-gravite-accidents",
    version=registered.version,
    stage="Staging"
)

print(f"✅ Modèle v{registered.version} enregistré en Staging !")

In [ ]:
# Vérification : recharger le modèle depuis le Registry et faire une prédiction
model_loaded = mlflow.pyfunc.load_model(
    model_uri="models:/xgboost-gravite-accidents/Staging"
)

y_pred_test = model_loaded.predict(X_test[:5])
print(f"✅ Modèle rechargé depuis le Registry !")
print(f"   Prédictions test : {y_pred_test}")

---
## 9. Comparaison finale des approches

Synthèse des résultats obtenus avec chaque approche de tuning.

In [ ]:
# Récupérer le meilleur run de chaque approche
def get_best_run(experiment_names, metric="metrics.recall"):
    """Retourne le meilleur run d'une liste d'expériences selon une métrique."""
    return mlflow.search_runs(
        experiment_names=experiment_names,
        order_by=[f"{metric} DESC"],
        filter_string=f"{metric} > 0"
    ).iloc[0]

best_baseline   = get_best_run(["accidents-gravite-baseline"])
best_tuning     = get_best_run(["accidents-gravite-tuning"])
best_gridsearch = get_best_run(["accidents-gravite-gridsearch"])
best_optuna     = get_best_run(["accidents-gravite-optuna"])

# Tableau comparatif
print("\n📊 COMPARAISON FINALE DES APPROCHES DE TUNING")
print("=" * 60)
print(f"{'Approche':<25} {'Recall':<12} {'ROC AUC':<12} {'Nb essais'}")
print("-" * 60)
print(f"{'Baseline XGBoost':<25} {best_baseline['metrics.recall']:.4f}       {best_baseline['metrics.roc_auc']:.4f}       1")
print(f"{'Tuning manuel':<25} {best_tuning['metrics.recall']:.4f}       {best_tuning['metrics.roc_auc']:.4f}       3")
print(f"{'GridSearchCV':<25} {best_gridsearch['metrics.recall']:.4f}       {best_gridsearch['metrics.roc_auc']:.4f}       27")
print(f"{'Optuna':<25} {best_optuna['metrics.recall']:.4f}       {best_optuna['metrics.roc_auc']:.4f}       30")
print("=" * 60)

# Identifier le gagnant
scores = {
    "Baseline XGBoost": best_baseline['metrics.recall'],
    "Tuning manuel":    best_tuning['metrics.recall'],
    "GridSearchCV":     best_gridsearch['metrics.recall'],
    "Optuna":           best_optuna['metrics.recall'],
}
gagnant = max(scores, key=scores.get)
print(f"\n🏆 Meilleure approche : {gagnant} — recall={scores[gagnant]:.4f}")
print(f"\n💡 Conclusion : Optuna explore des plages continues et trouve des configurations")
print(f"   que GridSearch n'aurait jamais testées (ex: max_depth=9, n_estimators=495).")